<a href="https://colab.research.google.com/github/peremartra/optipfair/blob/main/examples/layer_importance_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OptiPFair Notebook Series – Example: Analyze Layers Importance.

![optiPfair Logo](https://github.com/peremartra/optipfair/blob/main/images/optiPfair.png?raw=true)


This notebook demonstrates how to use the `analyze_layer_importance` function from OptiPFair to identify which transformer layers in a model are the most redundant or have the least impact on the model's representations.

This analysis is a crucial first step for developing a data-driven **Depth Pruning** strategy.

##Recommended Environment

- **Platform**: [Google Colab](https://colab.research.google.com)  
- **Hardware**: GPU runtime (recommended: T4 or better for 1B–3B models)  
- **Dependencies**: Installed automatically in the first cell (optipfair, transformers, torch)

##by Pere Martra.

- [LinkedIn](https://www.linkedin.com/in/pere-martra)  
- [GitHub](https://github.com/peremartra)  
- [X / Twitter](https://x.com/peremartra)

---

> If you find this useful, please ⭐ the [repository](https://github.com/peremartra/optipfair) and share it!
---
If you want to use your favorite LLM to create code with optiPfair, you just need to provide it with the file: [**optipfair_llm_reference_manual.txt**](https://github.com/peremartra/optipfair/blob/main/optipfair_llm_reference_manual.txt), which contains all the necessary information for the LLM to become an expert in using the library.

## 1. Installation and Setup
First, we install the necessary libraries. We include `matplotlib` and `seaborn` to visualize the results.

In [1]:
!pip install -q transformers optipfair torch

In [2]:
import torch
import os
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from torch.utils.data import DataLoader
from optipfair import prune_model_depth, analyze_layer_importance


# Check device availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Using device: cuda
GPU: Tesla T4
GPU Memory: 14.6 GB


## Load Model

For this example we'll use a Qwen3-0.6B model with 28 transformer blocks. We'll identify the three that contribute least to the dataset used and remove them, creating a new 25-block model.


In [3]:
MODEL_NAME = 'Qwen/Qwen3-0.6B'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto"
)
model.eval()

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

## Load & Preparing Dataset

We're going to use a standard dataset to measure the importance of the different transformer blocks. The idea is that you use the dataset you're preparing the model for, this way you'll get a lighter model that will work correctly with your data.

In [4]:
RECOVERY_SAMPLES = 100
BATCH_SIZE = 8
MAX_LENGTH = 512

In [6]:
dataset = load_dataset(
    'Salesforce/wikitext',
    'wikitext-2-raw-v1',
    split=f'train[:{RECOVERY_SAMPLES}]'
)

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/validation-00000-of-00(…): reconstructing file:   0%|          |  0.00B /  657kB            

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [7]:
def prepare_dataset(dataset, text_field='text'):
  def tokenize_function(examples):
      if text_field in examples:
          texts = examples[text_field]
      elif 'sms' in examples:  # SMS dataset
          texts = examples['sms']
      elif 'text' in examples:
          texts = examples['text']
      else:
          texts = examples[list(examples.keys())[0]]  # First available field

      return tokenizer(
          texts,
          truncation=True,
          padding='max_length',
          max_length=MAX_LENGTH,
          return_tensors='pt'
      )

  tokenized = dataset.map(tokenize_function, batched=True, remove_columns=dataset.column_names)
  tokenized.set_format(type='torch', columns=['input_ids', 'attention_mask'])
  return DataLoader(tokenized, batch_size=BATCH_SIZE, shuffle=False)

In [8]:
dataloader = prepare_dataset(dataset)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

## Call to `analyze_layer_importance`

The `analyze_layer_importance` function only needs to receive the model and the dataloader it should work with.

It will analyze how the different blocks transform information and return a dictionary with the layers and their cosine distance.

A value of 1 indicates it modifies information in an important way and therefore it's an active block that contributes to the final result. A value of 0 or close to 0 indicates the block is passive and doesn't alter information when passing through it.

Blocks with values close to 0 are the candidates to be eliminated.

In [9]:
importance_scores = analyze_layer_importance(model, dataloader, show_progress=True)


Processing batches: 100%|██████████| 13/13 [00:07<00:00,  1.80it/s]


In [10]:
print(importance_scores)

{0: np.float64(0.942140923669705), 1: np.float64(0.13162774764574492), 2: np.float64(0.1257519951233497), 3: np.float64(0.12933756296451276), 4: np.float64(0.12235481005448562), 5: np.float64(0.1000292759675246), 6: np.float64(0.09108441609602708), 7: np.float64(0.08750334152808556), 8: np.float64(0.07564267745384803), 9: np.float64(0.07838646723673894), 10: np.float64(0.07483403499309833), 11: np.float64(0.06352904668221107), 12: np.float64(0.03380010678217961), 13: np.float64(0.03324339481500479), 14: np.float64(0.052369892597198486), 15: np.float64(0.04642668137183556), 16: np.float64(0.06314427577532254), 17: np.float64(0.06109136343002319), 18: np.float64(0.05937315409000103), 19: np.float64(0.0738828182220459), 20: np.float64(0.06758656868567833), 21: np.float64(0.0601996871141287), 22: np.float64(0.046039232840904824), 23: np.float64(0.033250744526202865), 24: np.float64(0.03127351173987755), 25: np.float64(0.02431171673994798), 26: np.float64(0.035379721568180963), 27: np.float

In [11]:
layers_2_remove = sorted(importance_scores.keys(), key=lambda x: importance_scores[x])[:3]
print (layers_2_remove)

[25, 24, 13]


## Alternative: Using PyTorch TensorDataset (v0.2.4+)

Starting from **OptiPFair v0.2.4**, `analyze_layer_importance` supports multiple DataLoader formats, including native PyTorch structures like `TensorDataset`.

This is particularly useful when you don't want to use HuggingFace's dataset utilities and prefer working directly with PyTorch tensors.

### Supported Formats:
- **Dictionary** (HuggingFace style): `{'input_ids': tensor, 'attention_mask': tensor}`
- **Tuple/List** (TensorDataset): `(input_ids, attention_mask, ...)`
- **Single Tensor**: Direct tensor treated as `input_ids`

Let's see an example using `TensorDataset`:

In [12]:
# Reload model for clean state
del model
gc.collect()
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto"
)
model.eval()

print("Model reloaded successfully")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Model reloaded successfully


In [13]:
# Prepare dataset using TensorDataset (tuple format)
from torch.utils.data import TensorDataset

# Tokenize texts manually
sample_texts = [item['text'] for item in dataset if len(item['text'].strip()) > 10][:RECOVERY_SAMPLES]
inputs = tokenizer(
    sample_texts,
    truncation=True,
    padding='max_length',
    max_length=MAX_LENGTH,
    return_tensors='pt'
)

# Create TensorDataset - batches will be tuples: (input_ids, attention_mask)
tensor_dataset = TensorDataset(inputs['input_ids'], inputs['attention_mask'])
tensor_dataloader = DataLoader(tensor_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Created TensorDataset with {len(tensor_dataset)} samples")
print(f"DataLoader batch format: tuple (input_ids, attention_mask)")

Created TensorDataset with 75 samples
DataLoader batch format: tuple (input_ids, attention_mask)


In [14]:
# Analyze layer importance with TensorDataset
# OptiPFair v0.2.4+ automatically handles tuple format
importance_scores_tensor = analyze_layer_importance(
    model,
    tensor_dataloader,
    show_progress=True
)

print("\\nLayer importance scores (TensorDataset format):")
print(importance_scores_tensor)

Processing batches: 100%|██████████| 10/10 [00:04<00:00,  2.45it/s]

\nLayer importance scores (TensorDataset format):
{0: np.float64(0.9450038352981209), 1: np.float64(0.13285915851593016), 2: np.float64(0.12968307733535767), 3: np.float64(0.13102962374687194), 4: np.float64(0.12445158958435058), 5: np.float64(0.10145977139472961), 6: np.float64(0.09222328066825866), 7: np.float64(0.08814635872840881), 8: np.float64(0.07584601640701294), 9: np.float64(0.07863301038742065), 10: np.float64(0.0745725929737091), 11: np.float64(0.0629725694656372), 12: np.float64(0.03308926820755005), 13: np.float64(0.03244051933288574), 14: np.float64(0.05278699398040772), 15: np.float64(0.04619158506393432), 16: np.float64(0.06262005567550659), 17: np.float64(0.06028217077255249), 18: np.float64(0.059379053115844724), 19: np.float64(0.0732008159160614), 20: np.float64(0.06624353528022767), 21: np.float64(0.058999210596084595), 22: np.float64(0.04513254761695862), 23: np.float64(0.03218986988067627), 24: np.float64(0.0307864248752594), 25: np.float64(0.02364383339881897), 

### Key Benefits of TensorDataset Support

✅ **No HuggingFace Dependencies**: Work with pure PyTorch structures  
✅ **Automatic Format Detection**: OptiPFair handles dict, tuple, list, or single tensor formats  
✅ **Positional Mapping**: Tuple elements map to `[0]=input_ids`, `[1]=attention_mask`, `[2]=token_type_ids`, etc.  
✅ **Backward Compatible**: Existing dict-based code continues to work without changes

This makes OptiPFair more flexible for users who prefer native PyTorch workflows or have custom dataset implementations.

### Prune model.

With the least important blocks identified, we call `prune_model_depth` and remove the blocks.

It will return the new model without the selected layers.

In [15]:
pruned_model = prune_model_depth(
        model=model,
        layer_indices=layers_2_remove,
        show_progress=True
    )

Removing layers: 100%|██████████| 28/28 [00:00<00:00, 235351.73it/s]


In [16]:
pruned_model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-24): 25 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

The new model has only 25 layers.

Now you could start a Knowledge Distillation process to recover the lost general capabilities and you would have a lighter model that would work perfectly for the specific task you're preparing it for.

### ➡️ [**Star OptiPFair on GitHub**](https://github.com/peremartra/optipfair)

---
You can also follow my work and new projects on:

* **[LinkedIn](https://www.linkedin.com/in/pere-martra/)**
* **[X / Twitter](https://twitter.com/PereMartra)**